In [0]:
# Dataset
display(dbutils.fs.ls("/databricks-datasets/definitive-guide/data/retail-data/all/"))

In [0]:
dbutils.fs.cp(
    "/databricks-datasets/definitive-guide/data/retail-data/all/",
    "/Volumes/catalog_s/default/data/sales/",
    recurse=True
)


In [0]:
dbutils.fs.mv(
    "dbfs:/Volumes/catalog_s/default/data/sales/online-retail-dataset.csv",
    "dbfs:/Volumes/catalog_s/default/data/sales/sales.csv"
)

In [0]:
display(dbutils.fs.ls("/Volumes/catalog_s/default/data/sales/"))

In [0]:

df = spark.read.csv(
    "dbfs:/Volumes/catalog_s/default/data/sales/sales.csv",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
df = spark.read.csv(
    path="dbfs:/Volumes/catalog_s/default/data/sales/sales.csv",
    inferSchema=True,
    header=True
)

df.repartition(16).write.format("delta").mode("overwrite").partitionBy("country").saveAsTable(
    "catalog_s.default.sales_delta_partitioned"
)

In [0]:
%sql
select * from catalog_s.default.sales_delta_partitioned

In [0]:
# Data at delta location
display(dbutils.fs.ls("/data/output/sales_delta/"))

In [0]:
display(dbutils.fs.ls("/data/output/sales_delta_partitioned/Country=Australia/"))

In [0]:
%sql
use catalog_s.default;
select * from sales_delta_partitioned where InvoiceNo = '576394' and country = 'Australia'

In [0]:
%sql

select min(invoiceno), max(invoiceno), _metadata.file_name from sales_delta_partitioned
group by _metadata.file_name
order by min(invoiceno)

In [0]:
spark.conf.set(
    "spark.sql.files.maxPartitionBytes",
    134217728
)

In [0]:
%sql
OPTIMIZE sales_delta_partitioned where country = 'Australia' ZORDER BY (InvoiceNo)

In [0]:
%sql

select min(invoiceno), max(invoiceno), _metadata.file_name from sales_delta
group by _metadata.file_name
order by min(invoiceno)

In [0]:
%sql

select country, min(invoiceno), max(invoiceno), _metadata.file_name from sales_delta
group by country, _metadata.file_name
order by country, min(invoiceno)